In [1]:
from bs4 import BeautifulSoup

In [2]:
import re
import os
import pickle
import requests
from bs4 import BeautifulSoup
from collections import deque

pattern = r"https?:\/\/(?:www\d*\.)?eecs\.berkeley\.edu(?:\/[^\s]*)?"

os.makedirs("crawled", exist_ok=True)

def get_matching_urls(html, base_url=None):
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/") and base_url:
            href = base_url.rstrip("/") + href
        if re.match(pattern, href):
            urls.append(href)
    return urls


def crawl_website(html, url):
    if "Forbidden" in html:
        return
    safe_name = re.sub(r"[^\w]", "_", url)
    with open(f"crawled/{safe_name}.html", "w") as f:
        f.write(html)


seed = "https://eecs.berkeley.edu"
visited = set()
queue = deque([seed])
max_pages = 10

while queue and len(visited) < max_pages:
    url = queue.popleft()
    if url in visited:
        continue
    visited.add(url)
    print(f"Visiting ({len(visited)}/{max_pages}): {url}")
    
    response = requests.get(url, timeout=5)
    crawl_website(response.text, url)

    new_urls = get_matching_urls(response.text, base_url=url)
    for u in new_urls:
        if u not in visited:
            queue.append(u)

print(f"\nDone. Crawled {len(visited)} pages.")

Visiting (1/10): https://eecs.berkeley.edu

Done. Crawled 1 pages.
